# DX702 Coding Quiz: Week 7

In [20]:
import pandas as pd
import numpy as np
import statsmodels.api as sm # For linear regression
import statsmodels.formula.api as smf

from sklearn.neighbors import NearestNeighbors
import numpy as np

# from sklearn.linear_model import LinearRegression

In [21]:

df_7 = pd.read_csv('https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/main/homework_7.1.csv')
df_7.drop(columns=['Unnamed: 0'], inplace = True)
print(df_7.info())
df_7.head(n = 10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X       10000 non-null  float64
 1   W       10000 non-null  float64
 2   Z       10000 non-null  float64
 3   Y       10000 non-null  float64
dtypes: float64(4)
memory usage: 312.6 KB
None


,X,W,Z,Y
0,1.137055,1.221768,0.327829,1.944532
1,-0.112905,0.465835,0.599650,0.655514
2,2.077755,1.795414,-0.063393,5.934411
3,0.456373,-0.512159,1.177413,-0.188064
4,-1.012402,0.080002,-0.275697,-0.533775
5,1.467674,-0.029920,0.155538,0.416849
6,-1.179811,-1.043383,-1.493910,-2.057473
7,0.632285,-0.448065,-0.602200,0.913497
8,0.279169,0.909736,-0.035663,2.656631
9,0.414663,0.068680,0.811900,-0.409498


### Question 1

Suppose that a process can be modeled via linear regression: 

    `W = np.random.normal(0, 1, (1000,))
    X = W + np.random.normal(0, 1, (1000,))
    Z = np.random.normal(0, 1, (1000,)) 
    Y = X + Z + W + np.random.normal(0, 1, (1000,))`


Which is closest to the correlation of ﻿X﻿ with the error term in the equation for ﻿Y﻿? 

Option A)
0.5

Option B)
0.75

Option C)
1.0

Option D)
-0.50

`Option E)
0`



In [22]:

np.random.seed(0)

# Generate the data with standard deviation of 1
W   = np.random.normal(0, 1, 1000)
X   = W + np.random.normal(0, 1, 1000)
Z   = np.random.normal(0, 1, 1000)

noise_y       = np.random.normal(0, 1, 1000)
Y           = X + Z + W + noise_y

# Model Y as a function of X and Z only, so the error term includes W and noise
# noise  = Y - (X + Z)

#  correlation between X and the error term
correlation = np.corrcoef(X, noise_y)

print(f"Correlation between X and the error term: {correlation}")

Correlation between X and the error term: [[1.         0.01814143]
 [0.01814143 1.        ]]


<font color='plum'>The correlation between X and the error term in the equation for Y is approximately 0.

Explanation:
The error term is defined as the part of Y not explained by the model:
`error = Y − (X + Z + W)`

Since the noise added to Y is independent of X, Z, and W, the correlation between X and the error term (which is just the noise) is approximately zero.

### Question 2:
If ﻿Y﻿ is written as depending on ﻿X﻿ and ﻿Z﻿ only, ﻿W﻿ is part of the error term. 

Which, then, is closest to the expected correlation of ﻿X﻿ with the error term in the equation for ﻿Y﻿? 

Option A)
0.75

`Option B)
0.50`

Option C)
1.0

Option D)
0

Option E)
-0.50

In [23]:
np.random.seed(0)

# Generate the data with standard deviation of 1
W   = np.random.normal(0, 1, 1000)
X   = W + np.random.normal(0, 1, 1000)
Z   = np.random.normal(0, 1, 1000)

noise       = np.random.normal(0, 1, 1000)
Y           = X + Z + W + noise

# Model Y as a function of X and Z only, so the error term includes W and noise
error_term  = Y - (X + Z)

#  correlation between X and the error term
correlation = np.corrcoef(X, error_term)

print(f"Correlation between X and the error term: {correlation}")


Correlation between X and the error term: [[1.         0.50092964]
 [0.50092964 1.        ]]


<font color='plum'>

The correlation between X and the error term when Y is modeled as a function of X and Z only is 0.50

### Question 3

In the data frame for homework_7.1.csv, control for W by regressing ﻿Y﻿ on ﻿X﻿ and ﻿Z﻿ at the following constant values of ﻿W﻿: -1, 0, and 1. 

(You cannot literally use a constant value of W, of course, or you will have only one data point! How will you manage this?) 

QUESTION: How is the coefficient of X behaving ?

`Option A)
increasing`

Option B)
decreasing

Option C)
staying about the same (say, within 0.2 or so) as ﻿W﻿ increases? 


In [24]:

# Define the ranges for W
w_ranges = {
    -1: (-1.5, -0.5),
    0: (-0.5, 0.5),
    1: (0.5, 1.5)
}

x_coefficients = {}

# Perform regression for each range of W
for w_val, (w_min, w_max) in w_ranges.items():
    # Filter dataframe for the current range of W
    df_subset = df_7[(df_7['W'] > w_min) & (df_7['W'] < w_max)]
    
    # Define the model
    X = df_subset[['X', 'Z']]
    Y = df_subset['Y']
    X = sm.add_constant(X)  # Add a constant to the model
    
    # Fit the regression model
    model = sm.OLS(Y, X).fit()
    
    # Store coefficient of X
    x_coefficients[w_val] = model.params['X']

# Print the coefficients of X for each W
print("Coefficients of X for different values of W:")
sorted_coeffs = {k: v for k, v in sorted(x_coefficients.items())}
for w_val, coef in sorted_coeffs.items():
    print(f"For W around {w_val}, the coefficient of X is: {coef:.4f}")

# Determine trend of the coefficient of X
# coeffs = list(sorted_coeffs.values())
# if len(coeffs) == 3:
#     if coeffs[2] > coeffs[1] and coeffs[1] > coeffs[0]:
#         trend = "increasing"
#     elif coeffs[2] < coeffs[1] and coeffs[1] < coeffs[0]:
#         trend = "decreasing"
#     else:
#         # Check if they are staying about the same (within 0.2)
#         if abs(coeffs[0] - coeffs[1]) <= 0.2 and abs(coeffs[1] - coeffs[2]) <= 0.2 and abs(coeffs[0] - coeffs[2]) <= 0.2:
#             trend = "staying about the same"
#         else:
#             trend = "not consistently increasing, decreasing, or staying the same."
# else:
#     trend = "not enough data points to determine a trend"

# print(f"\nThe coefficient of X is {trend} as W increases.")

# # Choose the option
# if trend == "increasing":
#     answer = "Option A) increasing"
# elif trend == "decreasing":
#     answer = "Option B) decreasing"
# elif trend == "staying about the same":
#     answer = "Option C) staying about the same (say, within 0.2 or so)"
# else:
#     answer = "None of the above options."

# print(f"\nConclusion: The answer is {answer}.")

Coefficients of X for different values of W:
For W around -1, the coefficient of X is: 0.9901
For W around 0, the coefficient of X is: 1.4860
For W around 1, the coefficient of X is: 1.9937


### Question 4:



In [25]:
# Function to generate autocorrelated error
def make_error(corr_const, num): 

    err = list() 

    prev = np.random.normal(0, 1) 

    for n in range(num): 

        prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1) 

        err.append(prev) 

    return np.array(err) 


Create a linear regression model that uses this function as the error for both 
1. the Treatment, X, and 
2. the Outcome, Y. 
    (You can use random normal error for any other covariates, if you have them.) 


As `corr_const` increases from 0.2 to 0.5 to 0.8, find: 
- (i) the standard deviation of the estimate of the X coefficient over many trials, and 
- (ii) the mean of the standard error estimate of the X coefficient over many trials. 

(Hint: don't forget to include an intercept in your regression)

QUESTION: When `corr_const` increases, then: 


Option A)
(i) and (ii) remain about the same

Option B)
The ratio (i) / (ii) decreases

`Option C)
The ratio (i) / (ii) increases`

Option D)
(i) and (ii) differ, but their ratio remains about the same



In [ ]:

# Function to run the simulation
def run_simulation(corr_const, num_trials=1000, num_samples=1000):
    X_coeff_estimates   = []
    X_se_estimates      = []
    
    for _ in range(num_trials):
        # Generate data
        X_error = make_error(corr_const, num_samples)
        Y_error = make_error(corr_const, num_samples)
        
        X = np.random.normal(0, 1, num_samples) + X_error
        Y = 2 * X + np.random.normal(0, 1, num_samples) + Y_error
        
        # Add intercept
        X = sm.add_constant(X)
        
        # Fit 
        model = sm.OLS(Y, X).fit()
        
        #  coefficient estimate and its standard error
        X_coeff_estimates.append(model.params[1])
        X_se_estimates.append(model.bse[1])
    
    # (i) standard deviation of the estimate of the X coefficient over many trials
    std_X_coeff = np.std(X_coeff_estimates)
    
    # (ii) the mean of the standard error estimate of the X coefficient over many trials
    mean_X_se = np.mean(X_se_estimates)
    
    return std_X_coeff, mean_X_se

#  simulations for different correlation constants
corr_consts = [0.2, 0.5, 0.8]
results = {}

for corr_const in corr_consts:
    std_X_coeff, mean_X_se = run_simulation(corr_const)
    results[corr_const] = (std_X_coeff, mean_X_se)

for corr_const, (std_X_coeff, mean_X_se) in results.items():
    print(f"corr_const = {corr_const}: std_X_coeff = {std_X_coeff:.4f}, mean_X_se = {mean_X_se:.4f}")



corr_const = 0.2: std_X_coeff = 0.0323, mean_X_se = 0.0317
corr_const = 0.5: std_X_coeff = 0.0318, mean_X_se = 0.0316
corr_const = 0.8: std_X_coeff = 0.0318, mean_X_se = 0.0317


<font color = 'plum'>
 This exercise is designed to **illustrate the impact of autocorrelated error terms** on regression estimates—specifically, how **temporal correlation in the error** affects the **variability and reliability of coefficient estimates** in linear regression.


### **Purpose of the Question**

The question is trying to show:

1. **How autocorrelation in error terms affects regression estimates.**
2. **Why standard error estimates might become misleading** when error terms are not independent.
3. **The difference between the actual variability of estimates** (standard deviation over trials) and the **model-reported uncertainty** (mean of standard errors).


```python
def make_error(corr_const, num): 
    err = list() 
    prev = np.random.normal(0, 1) 
    for n in range(num): 
        prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1) 
        err.append(prev) 
    return np.array(err)
```

This function generates **autocorrelated error** using a simple AR(1)-like process:

- `corr_const` controls how strongly each error term depends on the previous one.
- As `corr_const` increases, the error becomes **more autocorrelated** (i.e., more "sticky" over time).

---

### **What You're Asked to Do**

1. Use this error function to generate errors for both:
   - The treatment variable \( X \)
   - The outcome variable \( Y \)

2. Run many simulations of a linear regression model:
   - Include an intercept.
   - Estimate the coefficient for \( X \).

3. For each value of `corr_const` (0.2, 0.5, 0.8), compute:
   - **(i)** The standard deviation of the estimated \( \beta_X \) across trials.
   - **(ii)** The mean of the standard error reported by the regression model.

4. Compare the ratio \( \frac{(i)}{(ii)} \) across different levels of autocorrelation.

---

### **What It's Trying to Show You**

As **autocorrelation increases**:

- The **true variability** of the coefficient estimates (i.e., (i)) **increases**.
- But the **model-reported standard error** (i.e., (ii)) **may not increase proportionally**, because standard regression assumes **independent errors**.
- Therefore, the ratio \( \frac{(i)}{(ii)} \) **increases**, showing that the model **underestimates uncertainty** when errors are autocorrelated.

---

### Correct Insight

> **The ratio (i) / (ii) increases** as `corr_const` increases.

This shows that **autocorrelation violates a key assumption of OLS regression**—independent errors—and leads to **underestimated standard errors**, which can make your results look more confident than they really are.
